# Module 8 | Class 4 -- Docker Containerization

**Objective:** Package the Flask prediction API (from Class 3) into a Docker container so it can run consistently on any machine.

**Important:** Docker cannot run inside Google Colab. This notebook **prepares all the files** (Dockerfile, docker-compose.yml, etc.) and provides the commands to build and run locally.

Prerequisites:
- Docker installed on your machine ([docs.docker.com/get-docker](https://docs.docker.com/get-docker/))
- The `flask_api/` folder from Class 3 (model.joblib, scaler.joblib, app.py, etc.)

## 1. Setup

First, make sure we have the model files from Class 3. If you already ran that notebook, the files should be in `flask_api/`. Otherwise, we recreate them here.

In [ ]:
!pip install -q scikit-learn pandas joblib

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

os.makedirs('flask_api', exist_ok=True)

# Check if model already exists
if os.path.exists('flask_api/model.joblib'):
    print("Model files already exist from Class 3.")
else:
    print("Recreating model files...")
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    df = pd.read_csv(url)
    df_proc = df.copy()
    df_proc['TotalCharges'] = pd.to_numeric(df_proc['TotalCharges'], errors='coerce')
    df_proc['TotalCharges'].fillna(df_proc['TotalCharges'].median(), inplace=True)
    df_proc.drop('customerID', axis=1, inplace=True)
    df_proc['Churn'] = (df_proc['Churn'] == 'Yes').astype(int)
    for col in df_proc.select_dtypes(include='object').columns:
        df_proc[col] = LabelEncoder().fit_transform(df_proc[col])
    X = df_proc.drop('Churn', axis=1)
    y = df_proc['Churn']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
    model.fit(X_train_scaled, y_train)
    joblib.dump(model, 'flask_api/model.joblib')
    joblib.dump(scaler, 'flask_api/scaler.joblib')
    with open('flask_api/feature_names.json', 'w') as f:
        json.dump(X.columns.tolist(), f)
    print("Model files created.")

print("\nFiles in flask_api/:")
for f in sorted(os.listdir('flask_api')):
    print(f"  {f}")

## 2. Write app.py

Same Flask app from Class 3, with a `/health` endpoint added (useful for Docker health checks).

In [ ]:
%%writefile flask_api/app.py
"""Flask API for Telco Churn prediction (Docker-ready)."""

from flask import Flask, request, jsonify
import joblib
import json
import numpy as np

app = Flask(__name__)

# Load model artifacts at startup
model = joblib.load('model.joblib')
scaler = joblib.load('scaler.joblib')
with open('feature_names.json') as f:
    FEATURE_NAMES = json.load(f)


@app.route('/')
def home():
    return jsonify({
        'service': 'Telco Churn Prediction API',
        'endpoints': {
            '/health': 'GET - health check',
            '/predict': 'POST - predict churn',
            '/info': 'GET - model info'
        }
    })


@app.route('/health')
def health():
    return jsonify({'status': 'healthy', 'model_loaded': True})


@app.route('/info')
def info():
    return jsonify({
        'model_type': type(model).__name__,
        'n_features': len(FEATURE_NAMES),
        'features': FEATURE_NAMES
    })


@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        if 'features' not in data:
            return jsonify({'error': 'Missing "features" key'}), 400
        features = data['features']
        if len(features) != len(FEATURE_NAMES):
            return jsonify({'error': f'Expected {len(FEATURE_NAMES)} features, got {len(features)}'}), 400

        X = np.array(features).reshape(1, -1)
        X_scaled = scaler.transform(X)
        prediction = int(model.predict(X_scaled)[0])
        proba = model.predict_proba(X_scaled)[0]

        return jsonify({
            'prediction': prediction,
            'churn': bool(prediction),
            'churn_probability': round(float(proba[1]), 4),
            'probabilities': {
                'no_churn': round(float(proba[0]), 4),
                'churn': round(float(proba[1]), 4)
            }
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    print(f"Model: {type(model).__name__}, Features: {len(FEATURE_NAMES)}")
    app.run(host='0.0.0.0', port=5000)

## 3. Write requirements.txt

In [ ]:
%%writefile flask_api/requirements.txt
flask>=2.0
scikit-learn>=1.0
joblib>=1.0
numpy>=1.21

## 4. Write Dockerfile

A Dockerfile is a set of instructions for building a container image. Each line creates a layer.

In [ ]:
%%writefile flask_api/Dockerfile
# Base image: Python 3.10 slim (small footprint)
FROM python:3.10-slim

# Set working directory inside the container
WORKDIR /app

# Copy requirements first (for Docker layer caching)
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the application
COPY . .

# Expose port 5000
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:5000/health')" || exit 1

# Run the Flask app
CMD ["python", "app.py"]

### Dockerfile Breakdown

| Instruction | What it does |
|---|---|
| `FROM python:3.10-slim` | Start from a minimal Python image |
| `WORKDIR /app` | Set `/app` as the working directory |
| `COPY requirements.txt .` | Copy requirements first (caching optimization) |
| `RUN pip install ...` | Install Python dependencies |
| `COPY . .` | Copy all remaining files into the container |
| `EXPOSE 5000` | Document that the container listens on port 5000 |
| `HEALTHCHECK` | Docker will periodically check if the service is up |
| `CMD ["python", "app.py"]` | Default command when the container starts |

## 5. Write docker-compose.yml

Docker Compose makes it easy to build and run the container with a single command.

In [ ]:
%%writefile flask_api/docker-compose.yml
services:
  churn-api:
    build: .
    ports:
      - "5000:5000"
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:5000/health')"]
      interval: 30s
      timeout: 5s
      retries: 3

## 6. Write .dockerignore

Prevents unnecessary files from being copied into the Docker image.

In [ ]:
%%writefile flask_api/.dockerignore
__pycache__
*.pyc
.git
.gitignore
.env
*.ipynb
*.ipynb_checkpoints
venv/
.venv/

## 7. Verify All Files

In [ ]:
expected_files = [
    'app.py', 'requirements.txt', 'Dockerfile',
    'docker-compose.yml', '.dockerignore',
    'model.joblib', 'scaler.joblib', 'feature_names.json'
]

print("File check:")
print("=" * 50)
all_good = True
for f in expected_files:
    path = f'flask_api/{f}'
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK" if exists else "MISSING"
    if not exists:
        all_good = False
    print(f"  [{status:7s}] {f:<25s} {size:>8,} bytes")

print("=" * 50)
if all_good:
    print("All files present. Ready to build.")
else:
    print("Some files are missing. Rerun the cells above.")

## 8. Build and Run Instructions

Download the `flask_api/` folder from Colab, then run these commands in your terminal.

### Option A: Docker Compose (recommended)
```bash
cd flask_api

# Build and start
docker compose up --build

# (In another terminal) Test the API
curl http://localhost:5000/health
curl http://localhost:5000/info

# Stop
docker compose down
```

### Option B: Docker directly
```bash
cd flask_api

# Build the image
docker build -t churn-api .

# Run the container
docker run -p 5000:5000 churn-api

# (In another terminal) Test
curl http://localhost:5000/health

# Stop: Ctrl+C or
docker ps           # find container ID
docker stop <ID>
```

## 9. Test Commands

In [ ]:
# Generate test curl commands you can copy-paste into your terminal

with open('flask_api/feature_names.json') as f:
    features = json.load(f)

# Create a sample payload with dummy values
sample_values = [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 29.85, 29.85, 1, 0]
# Pad or truncate to match feature count
while len(sample_values) < len(features):
    sample_values.append(0)
sample_values = sample_values[:len(features)]

payload = json.dumps({"features": sample_values})

print("=" * 60)
print("TEST COMMANDS (run after docker compose up)")
print("=" * 60)
print()
print("# Health check")
print("curl http://localhost:5000/health")
print()
print("# Model info")
print("curl http://localhost:5000/info")
print()
print("# Predict")
print(f"curl -X POST http://localhost:5000/predict \\")
print(f"  -H 'Content-Type: application/json' \\")
print(f"  -d '{payload}'")

---
## TODO: Student Exercise

**Task:** Modify the Dockerfile and app to containerize your own model from the capstone project.

Steps:
1. Replace `model.joblib` with your own trained model
2. Update `feature_names.json` to match your model's features
3. Update the `app.py` if your model needs different preprocessing
4. Rebuild the Docker image and test

Write your modified files below.

In [ ]:
# TODO: Write your modified Dockerfile
# %%writefile my_project/Dockerfile

# Your code here


In [ ]:
# TODO: Write your modified app.py
# %%writefile my_project/app.py

# Your code here


In [ ]:
# TODO: Write the build and test commands you would use
# (as comments or print statements)
